# 10 - Host MicrosoftAgent as a Non-Azure API

This notebook shows how to host a Microsoft Agent Framework agent as a regular HTTP API that can run on any compute platform (local VM, Docker, Kubernetes, Cloud Run, ECS, on-prem).

The agent is wired to the Azure Learn MCP endpoint over HTTP.

## What you get
- FastAPI service with /healthz and /chat endpoints
- Agent lifecycle management during API startup/shutdown
- In-notebook API verification
- Optional curl and external server run

In [10]:
import importlib.util

required_modules = [
    "agent_framework",
    "azure.identity",
    "dotenv",
    "fastapi",
    "pydantic",
]

missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "Missing dependencies: "
        + ", ".join(missing)
        + ". Install them in your environment and rerun Cell 2."
    )

print("Dependency check passed.")

Dependency check passed.


## 1) Build API App (Direct Azure Learn MCP HTTP)

This notebook uses the Learn MCP endpoint directly:
- `https://learn.microsoft.com/api/mcp`

Set these environment variables before running Cell 4:
- `FOUNDRY_PROJECT_ENDPOINT` or `AZURE_AI_PROJECT_ENDPOINT`
- `FOUNDRY_MODEL_DEPLOYMENT_NAME` or `AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME`

Optional:
- `LEARN_MCP_URL` (default is the Learn MCP URL above)

In [11]:
import os
from contextlib import asynccontextmanager

from azure.identity.aio import DefaultAzureCredential
from dotenv import load_dotenv
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

from agent_framework.azure import AzureAIClient

try:
    from agent_framework import MCPStreamableHTTPTool
except Exception as exc:
    raise RuntimeError(
        "MCPStreamableHTTPTool is not available in this environment. "
        "Use agent-framework-core/azure-ai compatible with this notebook."
    ) from exc

load_dotenv(override=False)

PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT") or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("FOUNDRY_MODEL_DEPLOYMENT_NAME") or os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME")
MCP_URL = os.getenv("LEARN_MCP_URL", "https://learn.microsoft.com/api/mcp")

if not PROJECT_ENDPOINT:
    raise RuntimeError(
        "Set FOUNDRY_PROJECT_ENDPOINT or AZURE_AI_PROJECT_ENDPOINT before running this cell."
    )

if not MODEL_DEPLOYMENT_NAME:
    raise RuntimeError(
        "Set FOUNDRY_MODEL_DEPLOYMENT_NAME or AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME before running this cell."
    )


def create_learn_mcp_tool():
    base_kwargs = {
        "name": "learn_mcp_server",
        "description": "Direct MCP connection to Microsoft Learn MCP endpoint",
        "tool_name_prefix": "learn",
        "approval_mode": "never_require",
        "load_prompts": False,
    }

    # Different prerelease versions may use different URL parameter names.
    for url_param in ("url", "server_url", "endpoint"):
        try:
            kwargs = dict(base_kwargs)
            kwargs[url_param] = MCP_URL
            return MCPStreamableHTTPTool(**kwargs)
        except TypeError:
            continue

    raise RuntimeError(
        "Could not initialize MCPStreamableHTTPTool with URL argument. "
        "Tried url/server_url/endpoint."
    )


class ChatRequest(BaseModel):
    message: str


class ChatResponse(BaseModel):
    answer: str


@asynccontextmanager
async def lifespan(app: FastAPI):
    credential = DefaultAzureCredential()

    mcp_tool = create_learn_mcp_tool()

    client = AzureAIClient(
        project_endpoint=PROJECT_ENDPOINT,
        model_deployment_name=MODEL_DEPLOYMENT_NAME,
        credential=credential,
    )

    agent_cm = client.as_agent(
        name="LearnMCPApiAgent",
        instructions=(
            "You are an API-facing assistant. "
            "Use available learn-prefixed MCP tools when helpful. "
            "If a tool fails, explain clearly and give a best-effort fallback."
        ),
        tools=[mcp_tool],
    )

    agent = await agent_cm.__aenter__()

    app.state.credential = credential
    app.state.mcp_tool = mcp_tool
    app.state.agent_cm = agent_cm
    app.state.agent = agent

    try:
        yield
    finally:
        await app.state.agent_cm.__aexit__(None, None, None)
        await app.state.mcp_tool.close()
        await app.state.credential.close()


app = FastAPI(title="MicrosoftAgent Learn-MCP API", lifespan=lifespan)


@app.get("/healthz")
async def healthz():
    return {"ok": True}


@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest):
    try:
        result = await app.state.agent.run(req.message)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc

    answer = getattr(result, "text", None) or str(result)
    return ChatResponse(answer=answer)


print("FastAPI app created: app")
print("Next: run Cell 6 to verify /healthz and /chat from inside this notebook.")

FastAPI app created: app
Next: run Cell 6 to verify /healthz and /chat from inside this notebook.


## 2) Verify the API In-Notebook

This uses FastAPI `TestClient`, so you can verify the API without launching a separate server process.

In [12]:
from fastapi.testclient import TestClient

with TestClient(app) as client:
    health = client.get("/healthz")
    print("/healthz status:", health.status_code)
    print("/healthz body:", health.json())

    chat_payload = {
        "message": (
            "Use learn MCP tools to find one Microsoft Learn resource about Agent Framework "
            "and include the URL in your answer."
        )
    }
    chat = client.post("/chat", json=chat_payload)
    print("/chat status:", chat.status_code)

    if chat.status_code != 200:
        raise RuntimeError(f"/chat failed: {chat.text}")

    body = chat.json()
    print("/chat body:", body)

    if "answer" not in body or not body["answer"]:
        raise RuntimeError("/chat response did not include a non-empty answer field.")

print("Notebook verification succeeded.")

/healthz status: 200
/healthz body: {'ok': True}
/chat status: 200
/chat body: {'answer': "Here is a Microsoft Learn resource about the Microsoft Agent Framework:\n\n- **Title**: Microsoft Agent Framework Workflows Orchestrations - Handoff (programming-language-csharp)\n- **URL**: [Microsoft Agent Framework Workflows Orchestrations - Handoff](https://learn.microsoft.com/agent-framework/workflows/orchestrations/handoff#what-you'll-learn)"}
Notebook verification succeeded.


## 3) Hosting Docs (Run Outside Notebook)

Do not run `uvicorn.run(...)` inside Jupyter because notebooks already run an event loop.

Use one of these hosting options from a terminal:

### Option A: Run locally with Uvicorn

1. Move the FastAPI app code (Cell 4) into a Python module, for example `app.py`, and keep the variable name as `app`.
2. Start the service:

```bash
export FOUNDRY_PROJECT_ENDPOINT="..."
export FOUNDRY_MODEL_DEPLOYMENT_NAME="..."
export LEARN_MCP_URL="https://learn.microsoft.com/api/mcp"
uvicorn app:app --host 0.0.0.0 --port 8000
```

3. Verify:

```bash
curl -sS http://127.0.0.1:8000/healthz
curl -sS http://127.0.0.1:8000/chat \
  -H "Content-Type: application/json" \
  -d '{"message":"Use learn tools and answer briefly."}'
```

### Option B: Run with Docker

Example Dockerfile:

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY . /app
RUN pip install -r requirements.txt
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

Build and run:

```bash
docker build -t learn-mcp-api .
docker run --rm -p 8000:8000 \
  -e FOUNDRY_PROJECT_ENDPOINT="..." \
  -e FOUNDRY_MODEL_DEPLOYMENT_NAME="..." \
  -e LEARN_MCP_URL="https://learn.microsoft.com/api/mcp" \
  learn-mcp-api
```

---

## Course Complete

You have now covered the core Microsoft Agent Framework walkthrough end-to-end.

| Notebook | Concept |
|----------|--------|
| 01 | Creating agents, streaming responses |
| 02 | Function tools - agents that call code |
| 03 | Structured output - typed Pydantic responses |
| 04 | Sessions - multi-turn conversations with memory |
| 05 | Agent composition - agents as tools |
| 06 | Middleware - compliance, PII guard, timing |
| 07 | Workflows - explicit graph-based orchestration |
| 08 | Multi-agent patterns - sequential, concurrent, fan-out/fan-in, human-in-the-middle |
| 09 | MCP with Agents |
| 10 | Hosting an agent as an API (FastAPI + Learn MCP HTTP) |

**Where to go next:**
- [Agent Framework docs](https://learn.microsoft.com/agent-framework/overview/)
- [Microsoft Agent Framework repo + samples](https://github.com/microsoft/agent-framework)